# RAG Assignment: Baseline Retrieval vs Cross-Encoder Re-Ranking

**Language Models and Agents Assignemt**

**Document:** *Syllabus for Courses of Monsoon 2026*, IIIT Hyderabad (Ver. 3, 03-08-2026), 295 pages.

This notebook builds **two retrieval pipelines over the same chunks** of one long PDF and compares them, as the assignment asks.

```
                ┌──────────────── offline (once) ─────────────────┐
  syllabus.pdf ─► extract text ─► chunk by course ─► embed (bi-encoder) ─► NumPy index
                                                                              │
  question ─► embed ─► cosine with every chunk ─► TOP 10 candidates ◄─────────┘
                                                      │
          ┌───────────────────────────────────────────┴───────────────────────┐
          │ Pipeline 1 (baseline)                    Pipeline 2 (re-ranking)   │
          │ keep bi-encoder top 3                    cross-encoder scores all  │
          │                                          10 (query, chunk) pairs,  │
          │                                          re-sort, keep top 3       │
          └───────────────┬───────────────────────────────────┬───────────────┘
                          ▼                                   ▼
             SAME prompt ─► LLM ─► Answer 1      SAME prompt ─► LLM ─► Answer 2
```
| Component | What we use |
|---|---|
| Knowledge source | IIIT-H Monsoon 2026 syllabus PDF |
| Chunking | 100-word windows, 20-word overlap (same as the tutorial), split by course, course name added to every chunk |
| Bi-encoder (retriever) | `sentence-transformers/all-MiniLM-L6-v2` (384-dim) |
| Vector index | a NumPy matrix, exact search |
| Cross-encoder (re-ranker) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Generator (LLM) | `google/gemma-3-4b-it`, local on the GPU (same as the tutorial) |
| Prompt | grounded: only the passages, cite them, refuse otherwise |

**Why this document works:** it is dated August 2026, after the LLM's training data, so the model cannot answer from memory (we check this in Section 1). It has **124 courses written in the same template** (credits, prerequisites, assessment, policies), so many chunks look alike. That is exactly where a bi-encoder gets confused and a cross-encoder can help.

## 0. Setup
Colab already has PyTorch, Transformers, NumPy and pandas. We install the three extra libraries we need:
- `pypdf`: reads the text out of the PDF.
- `sentence-transformers`: loads the bi-encoder and the cross-encoder.
- `accelerate`: lets Transformers place the LLM on the GPU (`device_map`).

In [ ]:
!pip install -q pypdf sentence-transformers accelerate

In [ ]:
import os, re, time, textwrap, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from transformers.utils import logging as hf_logging
from sentence_transformers import SentenceTransformer, CrossEncoder
from pypdf import PdfReader
hf_logging.set_verbosity_error()

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)

# ---------------- configuration: every important choice lives here ----------------
PDF_PATH           = "/content/syllabus.pdf"
LLM_NAME           = "google/gemma-3-4b-it"                     # generator (slide 16), same as the tutorial
FALLBACK_LLM       = "Qwen/Qwen2.5-3B-Instruct"                 # used only if Gemma cannot load (no licence/token)
BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"   # retriever (slides 24, 27)
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker (slides 27-29)

CHUNK_SIZE    = 100   # words per chunk (same as the tutorial)
CHUNK_OVERLAP = 20    # words shared by neighbouring chunks (slide 31: overlap softens the cut)
N_CANDIDATES  = 10    # the assignment: retrieve the top 10 with the bi-encoder
TOP_K         = 3     # the assignment: send only 3 chunks to the LLM
MAX_NEW_TOKENS = 300  # upper limit on answer length (same as the tutorial)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU, then run all cells again.")

def wrap(text, width=110):
    return "\n".join(textwrap.fill(line, width) for line in str(text).splitlines())

GPU: Tesla T4


## 1. The generator (LLM), and why we need retrieval at all

We load the **same LLM as the tutorial**, `google/gemma-3-4b-it`, and run it **locally on the GPU**, with the same settings:

- `bfloat16` stores each weight in 16 bits instead of 32, halving the memory, exactly as in the tutorial.
- `do_sample=False` is **greedy decoding**: the model always picks the most likely next token, so the same prompt gives the same answer every time. That makes Answer 1 vs Answer 2 a fair comparison.
- `max_new_tokens=300`, as in the tutorial.
- The LLM is **frozen**. RAG never retrains it, it only changes what goes into the prompt (slide 19).

**Safety net:** Gemma needs the Hugging Face licence and `HF_TOKEN` secret (see "How to run"). If it cannot load, or produces empty output on your GPU, the cell automatically switches to `Qwen/Qwen2.5-3B-Instruct`, a similar small open model that needs no licence. The cell prints which model it ended up using: write that name in your report.

In [ ]:
# Give Hugging Face your token (needed for the gated Gemma model).
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception as e:
    print("No HF_TOKEN secret found:", type(e).__name__, "- Gemma may fail to load; the fallback model will be used.")

Logged in to Hugging Face.


In [ ]:
def load_llm(name, dtype):
    tok = AutoTokenizer.from_pretrained(name)
    try:                               # newer Transformers versions call the argument `dtype`
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, device_map=DEVICE)
    except TypeError:                  # older versions call it `torch_dtype`
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, device_map=DEVICE)
    model.eval()
    # Same generation settings as the tutorial: greedy decoding, at most 300 new tokens.
    model.generation_config = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                               eos_token_id=model.generation_config.eos_token_id)
    return tok, model

def generate(prompt):
    # Send one user message to the chat model and return its reply (identical to the tutorial).
    messages = [{"role": "user", "content": prompt}]
    # The chat template adds the special tokens the model expects ("user turn starts", "model turn starts").
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs)
    # The output contains the prompt followed by the answer; slice off the prompt part.
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Load the LLM (1-3 minutes the first time: it downloads ~8 GB). Run this before anything else.
try:
    DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
    tokenizer, llm = load_llm(LLM_NAME, DTYPE)
    test = generate("In one sentence, what is retrieval-augmented generation?")
    if len(test) < 5:
        raise RuntimeError(f"empty or broken output: {test!r}")
except Exception as e:
    print(f"Could not use {LLM_NAME} ({type(e).__name__}: {str(e)[:150]})")
    print(f"Switching to the fallback model {FALLBACK_LLM} ...")
    llm = None; tokenizer = None
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    LLM_NAME = FALLBACK_LLM
    DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
    tokenizer, llm = load_llm(LLM_NAME, DTYPE)
    test = generate("In one sentence, what is retrieval-augmented generation?")

print("LLM IN USE:", LLM_NAME, "| dtype:", llm.dtype, "| device:", llm.device)
print(test)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

LLM IN USE: google/gemma-3-4b-it | dtype: torch.bfloat16 | device: cuda:0
Retrieval-augmented generation (RAG) combines a large language model with an external knowledge source, allowing it to generate more accurate and contextually relevant responses by first retrieving relevant information before generating text.


### 1.1 Closed-book test: does the model already know this document?
 same model, no document to consult. If the model could answer these from memory, retrieval would not matter and our comparison would show nothing. The correct answers are from the PDF.

In [ ]:
closed_book_tests = [
    ("Who teaches Language Models and Agents (CL3.410) at IIIT Hyderabad in Monsoon 2026?", "Vasudeva Varma"),
    ("What is the L-T-P-C of Data Structures & Algorithms for Problem Solving (CS1.304) at IIIT Hyderabad, Monsoon 2026?", "3-0-2-6"),
    ("What are the prerequisites of Introduction to UAV Design (EC4.402) at IIIT Hyderabad?", "Linear Algebra, Laplace transform, Vector calculus"),
]
for q, truth in closed_book_tests:
    print("Q:       ", q)
    print("CORRECT: ", truth)
    print("MODEL:   ", wrap(generate(q))[:400])
    print("-" * 100)

Q:        Who teaches Language Models and Agents (CL3.410) at IIIT Hyderabad in Monsoon 2026?
CORRECT:  Vasudeva Varma
MODEL:    As of today, November 2, 2023, the instructor for CL3.410 (Language Models and Agents) at IIIT Hyderabad for
the Monsoon 2026 semester **has not yet been announced.**

IIIT Hyderabad typically announces the faculty teaching each course well in advance, usually around June/July
for the Monsoon semester.

**Here's how you can find out the instructor when the information is released:**

*   **IIIT Hy
----------------------------------------------------------------------------------------------------
Q:        What is the L-T-P-C of Data Structures & Algorithms for Problem Solving (CS1.304) at IIIT Hyderabad, Monsoon 2026?
CORRECT:  3-0-2-6
MODEL:    Okay, let's break down the likely L-T-P-C (Lecture-Tutorial-Practical-Course) breakdown for CS1.304 – Data
Structures & Algorithms for Problem Solving at IIIT Hyderabad, Monsoon 2026.

**Based on past courses and typ

## 2. Load the PDF (the knowledge source)

We read the text layer of every page with `pypdf`. The syllabus is a real text PDF (not a scan), so no OCR is needed. Expect about **295 pages and ~98,000 words**. Extraction takes about 30 seconds. A warning like `incorrect startxref pointer` is harmless: the PDF file has a small formatting flaw that pypdf repairs on its own.

In [ ]:
if not os.path.exists(PDF_PATH):
    # The PDF is not in /content yet, so open Colab's upload box.
    from google.colab import files
    print("Choose the syllabus PDF to upload...")
    uploaded = files.upload()
    os.rename(next(iter(uploaded)), PDF_PATH)

reader = PdfReader(PDF_PATH)
pages = [p.extract_text() or "" for p in reader.pages]      # pages[0] is page 1
print(f"{len(pages)} pages, {sum(len(p.split()) for p in pages):,} words")
print("\n--- start of page 6 ---\n", pages[5][:700])

295 pages, 97,795 words

--- start of page 6 ---
 126 HS0.216 Thinking and Knowing in the 
Human Sciences - III 
3-1-0-4 Anirban Dasgupta + Khaliq 
Parkar 
127 CS9.442 Topics in AI for Healthcare 3-1-0-4 Vinod PK + Bapi Raju S + Raja 
Poladi 
128 SC2.401 Topics in Nanosciences 3-1-0-4 Tapan Kumar Sau 
129 CS6.501 Topics in Software Engineering 3-1-0-4 Raghu Reddy Y  
130 HS3.402 Understanding Public and Digital 
History 
3-1-0-4 Isha Dubey 
131 OC3.101 Value Education-1 (H) 0-2-0-2 Indranil Chakrabarty + Saurabh 
Todariya (Coordinators) 
132 EC2.201 VLSI Design 3-1-0-4 Abhishek Srivastava 
133 EC5.407 Wireless Communications 3-1-0-4 Rajeev Gangula 
 
Title of the Course  :  Advanced Computer Architecture 
Name of the Faculty  :  Suresh Puri


## 3. Chunking


1. **Structure aware.** Every course in this PDF starts with the line `Title of the Course : ...`. We first split the document **into courses**, so one chunk never mixes the end of one course with the start of the next.
2. **Fixed size with overlap inside each course.** 100-word windows (the tutorial's setting), with 20 words shared by neighbours, so a sentence cut at a boundary still appears whole in one of the two chunks.
3. **Metadata on every chunk.** Each chunk keeps its course name, course code and page number, for citations. We also **write the course name at the start of the chunk text**.
4. We also drop chunks that are mostly numbers. The CO-PO mapping tables extract as rows like `3 3 2 1 1 1 1 ...`, which carry no meaning and only add noise.


In [ ]:
def clean(t):
    return re.sub(r"\s+", " ", t).strip()

def split_into_courses(pages):
    """Group the PDF text into sections: one for the course list (pages 1-6), then one per course.
    Each section remembers which page every piece of its text came from."""
    sections = []
    cur = {"title": "Course list (all courses, codes, credits, faculty)", "code": "LIST", "parts": []}
    for pno, txt in enumerate(pages, start=1):
        # A page can contain the end of one course and the start of the next, so split the page too.
        for piece in re.split(r"(?=Title of the Course)", txt):
            if piece.startswith("Title of the Course"):
                if cur["parts"]:
                    sections.append(cur)
                m = re.match(r"Title of the Course\s*:?\s*(.+)", piece)
                title = clean(m.group(1)) if m else "Unknown course"
                title = re.split(r"\s{2,}|Name of the Faculty|Faculty", title)[0].strip(" :")[:90]
                cm = re.search(r"Course\s*[Cc]ode\s*:?\s*([A-Z]{2}\d\.\d{3}[a-z]?)", piece)
                cur = {"title": title, "code": cm.group(1) if cm else "", "parts": []}
            cur["parts"].append((pno, piece))
    sections.append(cur)
    return sections

def numeric_ratio(text):
    toks = text.split()
    return sum(t.replace(".", "").isdigit() for t in toks) / max(len(toks), 1)

def build_chunks(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, add_header=True):
    chunks = []
    for sec in split_into_courses(pages):
        words, word_page = [], []                     # keep the page number of every word
        for pno, piece in sec["parts"]:
            w = piece.split()
            words += w
            word_page += [pno] * len(w)
        step = size - overlap
        for start in range(0, max(len(words), 1), step):
            body = " ".join(words[start:start + size])
            if len(body.split()) >= 15 and numeric_ratio(body) < 0.5:   # skip tiny / number-only chunks
                code_txt = f" ({sec['code']})" if sec["code"] else ""
                header = f"[Course: {sec['title']}{code_txt}] " if add_header else ""
                chunks.append({"id": len(chunks), "course": sec["title"], "code": sec["code"],
                               "page": word_page[start], "text": header + body})
            if start + size >= len(words):
                break
    return chunks

sections = split_into_courses(pages)
chunks = build_chunks(pages)
print(f"{len(sections)} sections (1 course list + {len(sections) - 1} courses) -> {len(chunks)} chunks")
print(f"words per chunk: mean {np.mean([len(c['text'].split()) for c in chunks]):.0f}, "
      f"max {max(len(c['text'].split()) for c in chunks)}")

for c in chunks[300:302]:                              # two neighbouring chunks: notice the overlap
    print(f"\n--- chunk {c['id']} | {c['course']} | page {c['page']}\n{wrap(c['text'])}")

124 sections (1 course list + 123 courses) -> 1128 chunks
words per chunk: mean 102, max 111

--- chunk 300 | Design for Social Innovation | page 73
[Course: Design for Social Innovation (CS9.429)] in each month. Each module will address few topics through
lectures, chance to meet the social change makers through guest lectures, upclose work with NGOs and in
parallel work on the course project. MODULE [I] Social Immersion  What Is Social Innovation  Interaction
With Leading Changemakers  Identifying Core Developmental Sectors [Agri, Fintech, Healthcare, Edu etc] &
Field Visits  Student Paper [I] Case Study On A Social Innovation Of Choice  Hackathon #1: Needs
Identification (maybe, during a field immersion/at an NGO) MODULE [II] Societal Challenge Identification 
Deep Dive Into Defining The Problem  Market Research/Competitive Landscape

--- chunk 301 | Design for Social Innovation | page 73
[Course: Design for Social Innovation (CS9.429)] field immersion/at an NGO) MODULE [II] 

## 4. Bi-encoder and the vector index

A **bi-encoder** turns each text into one vector, **on its own**. The query and a chunk never see each other inside the network . They only meet at the end, when we compare the two vectors with cosine similarity:

$$\text{sim}(q, d) = \cos(q, d) = \frac{q \cdot d}{\lVert q\rVert\,\lVert d\rVert}$$

If every vector is first scaled to length 1 (`normalize_embeddings=True`), the denominator is 1 and cosine similarity is just a **dot product** $q^\top d$. Then scoring all chunks at once is one matrix-vector product.

**How it learned to do this:** the encoder was trained with a contrastive loss that pulls a query towards a correct passage $p^+$ and pushes it away from wrong passages $p^-_j$:

$$L = -\log\frac{\exp(\text{sim}(q,p^+))}{\exp(\text{sim}(q,p^+)) + \sum_{j}\exp(\text{sim}(q,p_j^-))}$$

**Indexing is done once, offline :** we embed every chunk and stack the vectors into a matrix of shape (number of chunks × 384). That matrix *is* our vector index . With ~1,000 chunks, exact search is instant. With millions you would use ANN search such as HNSW or IVF. It is the same operation, only faster and approximate.



In [ ]:
embedder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

# Important limitation: MiniLM reads at most `max_seq_length` tokens and silently ignores the rest.
lengths = [len(embedder.tokenizer(c["text"])["input_ids"]) for c in chunks]
print(f"bi-encoder max_seq_length = {embedder.max_seq_length} tokens")
print(f"chunk length in tokens: mean {np.mean(lengths):.0f}, max {max(lengths)}; "
      f"{sum(l > embedder.max_seq_length for l in lengths)} of {len(chunks)} chunks get truncated")

t0 = time.perf_counter()
index = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                        batch_size=64, convert_to_numpy=True, show_progress_bar=True)
print(f"index shape: {index.shape}  (one 384-number vector per chunk), built in {time.perf_counter() - t0:.1f} s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

bi-encoder max_seq_length = 256 tokens
chunk length in tokens: mean 167, max 341; 39 of 1128 chunks get truncated


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

index shape: (1128, 384)  (one 384-number vector per chunk), built in 3.5 s


## 5. The shared prompt

The assignment says both pipelines must use the **identical prompt structure**, so the *only* difference between Answer 1 and Answer 2 is which 3 chunks were retrieved. This template has five parts:

| Part | Name | Why |
|---|---|---|
| "Use ONLY the numbered passages" | Instruction | stops the model from answering from its own memory |
| `[1] (course, page) text...` | Evidence, delimited and labelled | so the model can point to a source |
| "Cite the passage number after every claim" | Citation rule | so *you* can check each claim |
| "If the passages do not contain the answer, say..." | Refusal clause | without it, a model handed the wrong passages makes something up |
| Question at the end | Question, last | the model reads the question right before it answers |

In [ ]:
PROMPT_TEMPLATE = """You are a helpful assistant answering questions about the IIIT Hyderabad Monsoon 2026 course syllabus.
Answer the question using ONLY the numbered passages below.
After every claim, cite the passage number you used, like [1] or [2].
If the passages do not contain the answer, reply exactly: "The provided passages do not contain the answer."

Passages:
{context}

Question: {question}
Answer:"""

def build_prompt(question, hits):
    context = "\n\n".join(f"[{i + 1}] (course: {h['course']}, page {h['page']})\n{h['text']}"
                          for i, h in enumerate(hits))
    return PROMPT_TEMPLATE.format(context=context, question=question)

def answer(question, hits):
    """Same prompt, same model, same decoding for both pipelines. Only `hits` changes."""
    return generate(build_prompt(question, hits))

## 6. Pipeline 1: baseline dense retrieval

Steps from the assignment:
- **(a)** chunks are already indexed with the bi-encoder (Section 4);
- **(b)** retrieve the **top 10** chunks by cosine similarity;
- **(c)** give the **top 3** directly to the LLM → **Answer 1**.

In [ ]:
def baseline_retrieve(question, n=N_CANDIDATES):
    """Bi-encoder: embed the question once, score every chunk with one dot product, keep the best n."""
    q = embedder.encode(question, normalize_embeddings=True, convert_to_numpy=True)
    scores = index @ q                       # (num_chunks x 384) @ (384,) -> one cosine score per chunk
    top = np.argsort(-scores)[:n]            # chunk positions, highest score first
    return [{**chunks[i], "bi_rank": r + 1, "bi_score": float(scores[i])} for r, i in enumerate(top)]

def pipeline1(question):
    candidates = baseline_retrieve(question)
    hits = candidates[:TOP_K]                # baseline: just trust the bi-encoder order
    return answer(question, hits), hits, candidates #answer function asks the llm and gets the ans back

def show_hits(hits, score_key="bi_score", n_chars=110):
    for h in hits:
        body = h["text"].split("] ", 1)[-1]  # hide the [Course: ...] header in previews
        print(f"[{h[score_key]:7.3f}] {h['course'][:38]:38s} p{h['page']:<4} {body[:n_chars]}...")

demo_q = "What is the weightage of the mid-semester exam in Introduction to Sociology?"
ans1, hits1, cands = pipeline1(demo_q)
print("Bi-encoder top 10:"); show_hits(cands)
display(Markdown(f"**Q: {demo_q}**\n\n**Answer 1 (baseline):** {ans1}"))

Bi-encoder top 10:
[  0.719] Introduction to Sociology              p150  through lectures. Relevant photographs, short 4-5 minutes videos, short films and clips from films are used du...
[  0.699] Introduction to Sociology              p150  (4 to 5 sentences): Type of Evaluation Weightage (in %) Mid Sem- Exam 15%. Questions designed to evaluate unde...
[  0.655] Thinking and Knowing in the Human Scie p275  and to draw interconnections between the different ways in which the human world is understood and explained. ...
[  0.619] Probability and Random Processes       p208  brief (4 to 5 sentences): 20 9 Type of Evaluation Weightage (in %) Mid Semester Exam 15% Quizzes/Viva 35% End ...
[  0.589] Thinking and Knowing in Human Sciences p277  Close, and Osvaldo Croci: Politics, Power and the Common Good: An Introduction to Political Science • Jonathan...
[  0.589] Structural Engineering Design Studio   p265  student and hence the assessment is divided as follows i.e.. a) 20% weightage is 

**Q: What is the weightage of the mid-semester exam in Introduction to Sociology?**

**Answer 1 (baseline):** The mid-semester exam has a weightage of 15%. [2]

## 7. Pipeline 2: cross-encoder re-ranking

A **cross-encoder** reads the query and a chunk **together, as one input**, so every query token can attend to every chunk token. It outputs one relevance score for the pair. It can notice things a bi-encoder blurs: *which* course is named, whether a sentence says "mandatory" or "not mandatory", and whether a number in the question matches a number in the chunk.

The price: nothing can be computed in advance, because the score depends on the pair. Every candidate needs its own forward pass at query time. So we only run it on the **10 candidates** from Pipeline 1, never on the whole document.

Steps from the assignment:
- **(a)** take the same top 10 candidates from Pipeline 1;
- **(b)** score all 10 (query, chunk) pairs with the cross-encoder;
- **(c)** re-sort by cross-encoder score and keep the **top 3**;
- **(d)** send them to the LLM with the **identical prompt** → **Answer 2**.

The division of labour is: **recall is set at retrieval** (is the right chunk anywhere in the top 10?), **precision is set at re-ranking** (is it in the top 3?). The re-ranker can only reorder what it is given: if the right chunk is not in the top 10, Pipeline 2 cannot rescue it.


In [ ]:
reranker = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

def rerank(question, candidates):
    """Score every (question, chunk) pair jointly and re-sort, highest score first."""
    ce_scores = reranker.predict([(question, c["text"]) for c in candidates], batch_size=16,
                                 show_progress_bar=False)
    order = np.argsort(-ce_scores)
    return [{**candidates[i], "ce_rank": r + 1, "ce_score": float(ce_scores[i])} for r, i in enumerate(order)]

def pipeline2(question, candidates=None):
    if candidates is None:                   # normally we pass Pipeline 1's top 10, so both use the same pool
        candidates = baseline_retrieve(question)
    reranked = rerank(question, candidates)
    hits = reranked[:TOP_K]
    return answer(question, hits), hits, reranked

ans2, hits2, reranked = pipeline2(demo_q, cands)
print("After cross-encoder re-ranking:"); show_hits(reranked, score_key="ce_score")
display(Markdown(f"**Q: {demo_q}**\n\n**Answer 2 (re-ranked):** {ans2}"))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

After cross-encoder re-ranking:
[  4.948] Introduction to Sociology              p150  (4 to 5 sentences): Type of Evaluation Weightage (in %) Mid Sem- Exam 15%. Questions designed to evaluate unde...
[  3.749] Introduction to Sociology              p150  through lectures. Relevant photographs, short 4-5 minutes videos, short films and clips from films are used du...
[  0.916] Probability and Random Processes       p208  brief (4 to 5 sentences): 20 9 Type of Evaluation Weightage (in %) Mid Semester Exam 15% Quizzes/Viva 35% End ...
[ -0.544] Thinking and Knowing in the Human Scie p275  and to draw interconnections between the different ways in which the human world is understood and explained. ...
[ -2.010] Thinking and Knowing in Human Sciences p277  Close, and Osvaldo Croci: Politics, Power and the Common Good: An Introduction to Political Science • Jonathan...
[ -2.914] Structural Engineering Design Studio   p265  student and hence the assessment is divided as follows i.e.. a) 20% 

**Q: What is the weightage of the mid-semester exam in Introduction to Sociology?**

**Answer 2 (re-ranked):** The weightage of the mid-semester exam in Introduction to Sociology is 15% [1].

## 8. The comparison tool

`compare()` runs both pipelines on one question and prints everything the report needs:

1. a table of the 10 candidates with their **bi-encoder rank, cross-encoder rank and both scores**, with the gold chunk marked ★;
2. the **rank shifts** in plain words, for example "chunk 452 moved #7 → #1";
3. **Answer 1 and Answer 2** (and, optionally, the closed-book answer with no retrieval at all).

In the table, `shift = bi_rank - ce_rank`: a **positive** shift means the cross-encoder moved that chunk **up** (for example +6 is #7 → #1), and a negative shift means it moved down.

**What is a "gold" chunk?** It is the chunk that truly contains the answer. We identify it by a short exact phrase from the PDF (`gold_text`) and, where useful, the course code (`gold_code`). We picked these phrases by reading the PDF, which is how you know what "correct" means.

In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", s).lower()

def is_gold(chunk, gold_text=None, gold_code=None):
    if not gold_text:
        return False
    ok = norm(gold_text) in norm(chunk["text"])
    return ok and (gold_code is None or chunk["code"] == gold_code)

def full_bi_rank(question, gold_text, gold_code=None):
    """Rank of the best gold chunk among ALL chunks (tells us if a miss was a recall failure)."""
    q = embedder.encode(question, normalize_embeddings=True, convert_to_numpy=True)
    order = np.argsort(-(index @ q))
    for r, i in enumerate(order):
        if is_gold(chunks[i], gold_text, gold_code):
            return r + 1
    return None

def compare(question, gold_text=None, gold_code=None, run_llm=True, closed_book=False):
    candidates = baseline_retrieve(question)
    reranked = rerank(question, candidates)
    ce_rank = {c["id"]: c for c in reranked}

    rows = []
    for c in candidates:
        r = ce_rank[c["id"]]
        rows.append({"chunk": c["id"], "gold": "★" if is_gold(c, gold_text, gold_code) else "",
                     "bi_rank": c["bi_rank"], "ce_rank": r["ce_rank"],
                     "shift": c["bi_rank"] - r["ce_rank"],
                     "bi_score": round(c["bi_score"], 3), "ce_score": round(r["ce_score"], 2),
                     "course": c["course"][:34], "page": c["page"],
                     "preview": c["text"].split("] ", 1)[-1][:70]})
    table = pd.DataFrame(rows).set_index("chunk")

    display(Markdown(f"### Q: {question}"))
    display(table)

    # Rank shifts in words: which chunks entered or left the top 3?
    for _, row in table.reset_index().iterrows():
        moved_in  = row.bi_rank > TOP_K and row.ce_rank <= TOP_K
        moved_out = row.bi_rank <= TOP_K and row.ce_rank > TOP_K
        if moved_in or moved_out or row.gold:
            tag = "promoted INTO the top 3" if moved_in else "dropped OUT of the top 3" if moved_out else "gold chunk"
            print(f"  chunk {row.chunk} ({row.course}, p{row.page}){' ★' if row.gold else ''}: "
                  f"#{row.bi_rank} -> #{row.ce_rank}  ({tag})")
    if gold_text and not table.gold.eq("★").any():
        print(f"  ★ gold chunk is NOT in the bi-encoder top {N_CANDIDATES} (full rank: "
              f"#{full_bi_rank(question, gold_text, gold_code)}). Re-ranking cannot fix a recall failure.")

    result = {"question": question, "table": table}
    if run_llm:
        if closed_book:
            result["answer0"] = generate(question)
            display(Markdown(f"**Answer 0 (no retrieval, closed book):** {result['answer0']}"))
        result["answer1"] = answer(question, candidates[:TOP_K])
        result["answer2"] = answer(question, reranked[:TOP_K])
        display(Markdown(f"**Answer 1 (baseline top 3):** {result['answer1']}\n\n"
                         f"**Answer 2 (re-ranked top 3):** {result['answer2']}"))
    return result

_ = compare(demo_q, gold_text="Mid Sem- Exam 15%", gold_code="HS2.201")

### Q: What is the weightage of the mid-semester exam in Introduction to Sociology?

,gold,bi_rank,ce_rank,shift,bi_score,ce_score,course,page,preview
chunk,,,,,,,,,
603,★,1,2,-1,0.719,3.75,Introduction to Sociology,150,"through lectures. Relevant photographs, short 4-5 minutes videos, shor"
604,★,2,1,1,0.699,4.95,Introduction to Sociology,150,(4 to 5 sentences): Type of Evaluation Weightage (in %) Mid Sem- Exam
1048,,3,4,-1,0.655,-0.54,Thinking and Knowing in the Human,275,and to draw interconnections between the different ways in which the h
800,,4,3,1,0.619,0.92,Probability and Random Processes,208,brief (4 to 5 sentences): 20 9 Type of Evaluation Weightage (in %) Mid
1056,,5,5,0,0.589,-2.01,Thinking and Knowing in Human Scie,277,"Close, and Osvaldo Croci: Politics, Power and the Common Good: An Intr"
1012,,6,6,0,0.589,-2.91,Structural Engineering Design Stud,265,student and hence the assessment is divided as follows i.e.. a) 20% we
708,,7,9,-2,0.586,-3.97,Modern Complexity Theory,182,presentation or a term paper. This will encourage self- exploration an
444,,8,10,-2,0.575,-5.14,Gender and Society,111,Press. 5.Teaching-Learning Strategies: Students are expected to read u
483,,9,7,2,0.566,-3.12,Intro to Cognitive Science,122,10% Debate+report or Term Paper (in class) 20% Mid semester exam 20% E


  chunk 603 (Introduction to Sociology, p150) ★: #1 -> #2  (gold chunk)
  chunk 604 (Introduction to Sociology, p150) ★: #2 -> #1  (gold chunk)
  chunk 1048 (Thinking and Knowing in the Human , p275): #3 -> #4  (dropped OUT of the top 3)
  chunk 800 (Probability and Random Processes, p208): #4 -> #3  (promoted INTO the top 3)


**Answer 1 (baseline top 3):** The mid-semester exam has a weightage of 15%. [2]

**Answer 2 (re-ranked top 3):** The weightage of the mid-semester exam in Introduction to Sociology is 15% [1].

## 9. Stress testing: a bank of candidate failure queries

The assignment asks for at least **two queries where the baseline fails and re-ranking fixes it**, plus document-specific failures of these types:

| Type | What it means | How it shows up in this syllabus |
|---|---|---|
| **Keyword trap / distractor** | the query shares many words with a wrong passage, while the true answer uses different words | 124 courses all have "Mid Sem Exam", "Assignments", "Prerequisites" sections. The wrong course's section looks almost the same. Also synonyms: "drones" vs "UAV", "rewarded for helping" vs "Karma Points" |
| **Exceptions & corner cases** | the general rule is retrieved, the "except when..." is missed | late work allowed only with instructor approval; a *second* plagiarism offence has a different penalty; an intake limit |
| **Negation & constraints** | "not allowed", "not mandatory", role limits | "can be useful but not mandatory"; "not asking another student for help debugging"; the giver of help is also punished |
| **Buried chunk** | the right chunk sits at rank 4-10 and must be promoted | numbers and table rows (credits `3-0-2-6`, "Attendance: 5%"), which bi-encoders handle poorly |

We cannot know in advance which queries will actually fail, so we test **all** of them and let the numbers decide. The sweep below does **not** call the LLM, so it takes only a few seconds.

**Add your own queries.** Read the PDF, find a fact, copy a short exact phrase from it as `gold_text`, and add a line. Queries you write and verify yourself are the strongest part of your report.

In [ ]:
#   (failure type, question, gold_text: exact phrase from the PDF, gold_code: course code or None)
QUERY_BANK = [
    # --- keyword trap / distractor -------------------------------------------------------------
    ("keyword trap", "What is the weightage of the mid-semester exam in Introduction to Sociology?",
        "Mid Sem- Exam 15%", "HS2.201"),
    ("keyword trap", "How are students in Information Retrieval and Extraction rewarded for helping other students in class?",
        "Karma Points", "CS4.406"),
    ("keyword trap", "Which course teaches students to design, build and flight-test their own drones?",
        "prototype UAV", "EC4.402"),
    ("keyword trap", "What are the prerequisites for Introduction to UAV Design?",
        "Laplace transform", "EC4.402"),
    # --- exceptions & corner cases ---------------------------------------------------------------
    ("exception", "Can I submit an assignment after the deadline in Intro to Cognitive Science?",
        "no make-up assignments", "CG1.402"),
    ("exception", "In Intro to Cognitive Science, what happens if plagiarism is detected a second time?",
        "one letter grade will be deducted", "CG1.402"),
    ("exception", "Is there a limit on how many students can register for Autonomous Software Engineering?",
        "In Take Limit", "CS6.403"),
    # --- negation & constraints ------------------------------------------------------------------
    ("negation", "Is Mathematics of Information a mandatory prerequisite for Information, Computation, and Learning?",
        "can be useful but not mandatory", "EC5.415"),
    ("negation", "Which prerequisite course is mandatory, not just desired, for Autonomous Software Engineering?",
        "(Mandatory)", "CS6.403"),
    ("negation", "In Introduction to Psychology, am I allowed to ask another student to help debug my assignment code?",
        "not asking another student for help debugging", "CG1.401"),
    ("negation", "Am I allowed to use AI or LLMs while writing the term paper in Intro to Cognitive Science?",
        "use AI/LLMs in ways that are not authorized", "CG1.402"),
    ("negation", "In Introduction to Psychology, is the student who gives help also penalised, or only the one who copies?",
        "giver or receiver", "CG1.401"),
    # --- buried chunks: numbers and table rows ----------------------------------------------------
    ("buried", "Which course has an L-T-P-C of 3-0-2-6?", "3-0-2-6", None),
    ("buried", "What percentage of the grade is attendance in Physics of Early Universe?", "Attendance: 5%", "SC1.415"),
    ("buried", "Who teaches Language Models and Agents?", "Vasudeva Varma", None),
]

def classify(bi, ce):
    if bi is None:            return "gold chunk not found (check gold_text)"
    if bi > N_CANDIDATES:     return "recall failure (not in top 10)"
    if bi <= TOP_K and ce <= TOP_K: return "already fine"
    if bi > TOP_K and ce <= TOP_K:  return "FIXED by re-ranking"
    if bi <= TOP_K and ce > TOP_K:  return "HURT by re-ranking"
    return "not fixed (still outside top 3)"

def gold_sweep(bank):
    rows = []
    for ftype, q, gold_text, gold_code in bank:
        cands = baseline_retrieve(q)
        rr = rerank(q, cands)
        bi = next((c["bi_rank"] for c in cands if is_gold(c, gold_text, gold_code)), None)
        ce = next((c["ce_rank"] for c in rr if is_gold(c, gold_text, gold_code)), None)
        if bi is None:                         # not in the top 10: find its true rank in the whole index
            bi = full_bi_rank(q, gold_text, gold_code)
        rows.append({"type": ftype, "question": q, "bi_rank": bi,
                     "ce_rank": ce if ce is not None else "-", "status": classify(bi, ce)})
    return pd.DataFrame(rows)

sweep = gold_sweep(QUERY_BANK)
display(sweep)
print(sweep.status.value_counts().to_string())

,type,question,bi_rank,ce_rank,status
0,keyword trap,What is the weightage of the mid-semester exam in Introduction to Sociology?,1,1,already fine
1,keyword trap,How are students in Information Retrieval and Extraction rewarded for helping other st...,1,2,already fine
2,keyword trap,"Which course teaches students to design, build and flight-test their own drones?",2,3,already fine
3,keyword trap,What are the prerequisites for Introduction to UAV Design?,2,1,already fine
4,exception,Can I submit an assignment after the deadline in Intro to Cognitive Science?,1,1,already fine
5,exception,"In Intro to Cognitive Science, what happens if plagiarism is detected a second time?",1,1,already fine
6,exception,Is there a limit on how many students can register for Autonomous Software Engineering?,5,1,FIXED by re-ranking
7,negation,"Is Mathematics of Information a mandatory prerequisite for Information, Computation, a...",3,1,already fine
8,negation,"Which prerequisite course is mandatory, not just desired, for Autonomous Software Engi...",2,1,already fine
9,negation,"In Introduction to Psychology, am I allowed to ask another student to help debug my as...",2,1,already fine


status
already fine                      13
FIXED by re-ranking                1
recall failure (not in top 10)     1


In [ ]:
# Harder queries: most do not name the course, or use different words from the PDF.
QUERY_BANK_2 = [
    ("keyword trap", "Which course limits enrollment to 30 students?", "In Take Limit", "CS6.403"),
    ("keyword trap", "In which course can a TA award bonus marks to students who help their classmates?", "Karma Points", "CS4.406"),
    ("keyword trap", "Is there a course where students build and fly their own drone?", "prototype UAV", "EC4.402"),
    ("keyword trap", "Which course has assignments that each need 30 to 40 pages of reading?", "30-40 pages of intense reading", "HS2.201"),
    ("exception", "Which course says there are no make-up quizzes and no late work without the instructor's approval?", "no make-up assignments", "CG1.402"),
    ("exception", "Which course lowers the final grade by one letter for a second plagiarism offence?", "one letter grade will be deducted", "CG1.402"),
    ("exception", "Can a BTech student take Behavioral Research and Experimental Design?", "Open only for DD, MS, and PhD students", "CG3.402"),
    ("exception", "In which course can repeated academic dishonesty make you fail the whole course?", "fail the course in case of repetition", "CG1.401"),
    ("negation", "Does Language Models and Agents have an end-semester exam?", "No End Sem exam", "CL3.410"),
    ("negation", "Which course has no mid-semester exam?", "No Mid Sem Exam", "CL2.203"),
    ("negation", "Which course replaces the end-semester exam with a mini-project demo?", "mini-project demo in the final week", "CL3.412"),
    ("negation", "Which course forbids using ChatGPT for written work unless the faculty allows it?", "use AI/LLMs in ways that are not authorized", "CG1.402"),
    ("negation", "Which course punishes both the student who shares an assignment and the one who copies it?", "giver or receiver", "CG1.401"),
    ("negation", "For the information theory course, is Mathematics of Information required beforehand?", "can be useful but not mandatory", "EC5.415"),
    ("buried", "Which course gives 5% of the total marks for attendance?", "Attendance: 5%", "SC1.415"),
    ("buried", "Which course uses an open book exam followed by a project with flight testing?", "Open book exam followed by detailed project", "EC4.402"),
]

sweep2 = gold_sweep(QUERY_BANK_2)
display(sweep2)
print(sweep2.status.value_counts().to_string())

# Merge both banks so Sections 10, 11 and 13 use all the queries.
QUERY_BANK = QUERY_BANK + QUERY_BANK_2
sweep = pd.concat([sweep, sweep2], ignore_index=True)

,type,question,bi_rank,ce_rank,status
0,keyword trap,Which course limits enrollment to 30 students?,411,-,recall failure (not in top 10)
1,keyword trap,In which course can a TA award bonus marks to students who help their classmates?,1,1,already fine
2,keyword trap,Is there a course where students build and fly their own drone?,2,2,already fine
3,keyword trap,Which course has assignments that each need 30 to 40 pages of reading?,140,-,recall failure (not in top 10)
4,exception,Which course says there are no make-up quizzes and no late work without the instructor...,1,1,already fine
5,exception,Which course lowers the final grade by one letter for a second plagiarism offence?,1,1,already fine
6,exception,Can a BTech student take Behavioral Research and Experimental Design?,1,1,already fine
7,exception,In which course can repeated academic dishonesty make you fail the whole course?,3,1,already fine
8,negation,Does Language Models and Agents have an end-semester exam?,1,1,already fine
9,negation,Which course has no mid-semester exam?,475,-,recall failure (not in top 10)


status
already fine                      9
recall failure (not in top 10)    6
FIXED by re-ranking               1


**How to read the sweep table:**
- `bi_rank`: where the gold chunk sits in the bi-encoder ranking (1 = best).
- `ce_rank`: where it sits after re-ranking the top 10 (`-` means it was never in the top 10).
- **FIXED by re-ranking**: the baseline sent the wrong chunks to the LLM, re-ranking sent the right one. These are your main failure queries.
- **recall failure**: neither pipeline can answer, because the right chunk never reached the top 10 (slide 29: recall is set at retrieval). Worth one honest sentence in your report.
- **HURT by re-ranking**: the cross-encoder made it worse. Also worth reporting, since re-rankers are not perfect.

If fewer than two queries are FIXED, reword some questions (for example, use the words of a *different* section, or ask about a number), or add your own.

## 10. Detailed failure analysis (rank shifts + Answer 1 vs Answer 2)

We now run the full comparison, **with the LLM**, on the queries the sweep marked FIXED (plus one HURT or recall-failure case if there is one, for balance). Each takes about 15 to 30 seconds because the LLM writes three answers.

For each query, look for:
- **rank shift:** which chunk moved into the top 3, and from where;
- **Answer 1:** did the baseline hallucinate, say it doesn't know, answer about the *wrong course*, or **omit a clause** ("except with instructor approval", "not mandatory")?
- **Answer 2:** is it correct, and does it cite the right passage?

In [16]:
fixed = sweep[sweep.status == "FIXED by re-ranking"]
others = sweep[sweep.status.isin(["HURT by re-ranking", "recall failure (not in top 10)"])].head(1)
selected = pd.concat([fixed, others])
if len(fixed) < 2:
    print(f"Only {len(fixed)} query is FIXED by re-ranking, and the assignment needs at least 2.\n"
          "Go back to Section 9: reword some questions or add your own to QUERY_BANK, re-run the sweep, "
          "then re-run this cell.\n")
print(f"Running full comparisons on {len(selected)} queries:\n")

bank_lookup = {q: (t, g, c) for t, q, g, c in QUERY_BANK}
RESULTS = []
for q in selected.question:
    ftype, gold_text, gold_code = bank_lookup[q]
    display(Markdown(f"---\n**Failure type: {ftype}**"))
    res = compare(q, gold_text, gold_code, run_llm=True, closed_book=True)
    res["type"] = ftype
    RESULTS.append(res)

Running full comparisons on 3 queries:



---
**Failure type: exception**

### Q: Is there a limit on how many students can register for Autonomous Software Engineering?

,gold,bi_rank,ce_rank,shift,bi_score,ce_score,course,page,preview
chunk,,,,,,,,,
125,,1,3,-2,0.506,-0.31,Autonomous Software Engineering (A,33,for software engineering tasks. Learning is reinforced through in-clas
126,,2,2,0,0.492,0.08,Autonomous Software Engineering (A,33,the course is a team- based project where students develop and evaluat
127,,3,4,-1,0.462,-0.56,Autonomous Software Engineering (A,33,and guided project work. 7. Assessment methods and weightages in brief
120,,4,5,-1,0.447,-2.78,Autonomous Software Engineering (A,32,2 2 3 2 3 2 2 2 3 2 2 2 2 2 3 2 4. Detailed Syllabus: Unit1: Foundatio
118,★,5,1,4,0.445,2.43,Autonomous Software Engineering (A,31,Title of the Course : Autonomous Software Engineering (AutoSE) Name of
1082,,6,9,-3,0.433,-8.25,Topics in Software Engineering,285,"as Artificial Intelligence for software engineering, self-adaptive sys"
257,,7,10,-3,0.427,-11.05,Computer Programming,63,"once in week. On the rest of the days, teaching assistants conduct off"
119,,8,7,1,0.419,-3.63,Autonomous Software Engineering (A,31,and agent-centric solutions for software engineering activities. CO3:
121,,9,8,1,0.412,-4.24,Autonomous Software Engineering (A,32,"Architecture tasks, Views and Viewpoint generation, architecture evalu"


  chunk 127 (Autonomous Software Engineering (A, p33): #3 -> #4  (dropped OUT of the top 3)
  chunk 118 (Autonomous Software Engineering (A, p31) ★: #5 -> #1  (promoted INTO the top 3)


...

### Q: Which course has an L-T-P-C of 3-0-2-6?

,gold,bi_rank,ce_rank,shift,bi_score,ce_score,course,page,preview
chunk,,,,,,,,,
1035,,1,6,-5,0.502,-7.84,Theory of Elasticity,272,(PSOs) – Course Articulation Matrix Note: Each Course Outcome (CO) may
472,,2,10,-8,0.477,-8.42,"Information, Computation, and Lear",118,participation in at least 5 tut sessions. Max of 10 marks) 10 Project
1057,,3,4,-1,0.467,-5.54,Thinking and Knowing in Human Scie,278,Term Paper Other Evaluation Mapping of Course Outcomes to Program Obje
345,,4,1,3,0.463,2.48,Discrete Structures,83,Title of the Course : Discrete Structures Name of the Faculty : Pravee
628,,5,5,0,0.456,-6.09,Language Models and Agents,158,No End Sem exam - Mapping of Course Outcomes to Program Objectives Cou
533,,6,8,-2,0.452,-8.11,Introduction to History,133,30 End Sem Exam 40 TOTAL 100 Mapping of Course Outcomes to Program Obj
347,,7,7,0,0.449,-7.91,Discrete Structures,84,2 2 2 2 2 2 2 1 1 2 3 CO5 1 1 1 1 1 1 2 2 1 1 1 1 1 1 2 3 CO6 2 2 2 2
1041,,8,2,6,0.432,1.92,Thinking and Knowing in the Human,273,Title of the Course: Thinking and Knowing in the Human Sciences – II N
944,,9,3,6,0.428,-5.00,Signal Processing,247,3 3 3 3 3 - - - 3 2 - 1 1 3 1 1 CO7 3 3 3 3 3 - - - 3 2 2 1 - 3 1 1 24


  chunk 1035 (Theory of Elasticity, p272): #1 -> #6  (dropped OUT of the top 3)
  chunk 472 (Information, Computation, and Lear, p118): #2 -> #10  (dropped OUT of the top 3)
  chunk 1057 (Thinking and Knowing in Human Scie, p278): #3 -> #4  (dropped OUT of the top 3)
  chunk 345 (Discrete Structures, p83): #4 -> #1  (promoted INTO the top 3)
  chunk 1041 (Thinking and Knowing in the Human , p273): #8 -> #2  (promoted INTO the top 3)
  chunk 944 (Signal Processing, p247): #9 -> #3  (promoted INTO the top 3)
  ★ gold chunk is NOT in the bi-encoder top 10 (full rank: #424). Re-ranking cannot fix a recall failure.


**Answer 0 (no retrieval, closed book):** Okay, let's break down what an L-T-P-C of 3-0-2-6 means and then figure out which course it likely represents.

*   **L** = Lectures
*   **T** = Tutorials/Practical Sessions
*   **P** = Practical/Laboratory Sessions
*   **C** = Credits

So, a course with an L-T-P-C of 3-0-2-6 means:

*   **3** Lectures
*   **0** Tutorials
*   **2** Practical Sessions
*   **6** Credits

**This type of structure is very common in university courses, particularly in engineering, science, and some humanities fields.**

**Without knowing the specific university or department, it's impossible to say *exactly* which course it is.** However, it's likely a course that combines theoretical instruction (lectures) with hands-on application (practical sessions) and is worth a significant amount of academic credit.

**To help me narrow it down and give you a more specific answer, could you tell me:**

*   **What university or institution are you asking about?** (e.g., University of Toronto, MIT, etc.)
*   **What field of study are you interested in?** (e.g., Computer Science, Biology, Physics, History, etc.)

**Answer 1 (baseline top 3):** The provided passages do not contain the answer.

**Answer 2 (re-ranked top 3):** The provided passages do not contain the answer.

### Your observations (write these yourself)

For each failure query above, write 2 or 3 lines in your report:

| Query | Rank shift of the gold chunk | What went wrong in Answer 1 | Did Answer 2 fix it? |
|---|---|---|---|
| *(fill in)* | e.g. #7 → #1 | e.g. answered about the wrong course / missed "not mandatory" | |

Also try a query of your own:

In [ ]:
# Try your own question. Put a short exact phrase from the PDF as gold_text so the gold chunk gets a ★.
_ = compare("Which courses in Monsoon 2026 have an intake limit?",
            gold_text="In Take Limit", gold_code=None, run_llm=True)

### Q: Which courses in Monsoon 2026 have an intake limit?

,gold,bi_rank,ce_rank,shift,bi_score,ce_score,course,page,preview
chunk,,,,,,,,,
82,,1,6,-5,0.498,-4.67,Advanced Probability Theory,22,gap between coursework and research-level engagement. ----------------
466,,2,1,1,0.447,0.98,"Information, Computation, and Lear",117,"Title of the Course : Information, Computation, and Learning Faculty N"
510,,3,3,0,0.436,0.29,Introduction to Economics,128,Title of the Course : Introduction to Economics Name of the Faculty :
411,,4,7,-3,0.405,-8.17,Environmental Science and Technolo,102,Title of the Course : Environmental Science and Technology Name of the
413,,5,9,-4,0.398,-11.25,Environmental Science and Technolo,103,2 2 2 2 2 CO6 2 2 2 2 2 2 2 2 1 1 1 2 2 1 2 2 4. Detailed Syllabus: Un
1041,,6,8,-2,0.395,-9.76,Thinking and Knowing in the Human,273,Title of the Course: Thinking and Knowing in the Human Sciences – II N
1050,,7,4,3,0.395,0.09,Thinking and Knowing in Human Scie,275,Title of the Course : Thinking and Knowing in Human Sciences- 3 Name o
264,,8,5,3,0.383,-1.84,Cyber GIS: Geospatial Web and GeoB,64,Title of the Course : Cyber GIS: Geospatial Web and GeoBI Name of the
874,,9,2,7,0.380,0.61,Readings from Hindi Literature,228,Title of the Course : Readings from Hindi Literature Faculty Name : Ha


  chunk 82 (Advanced Probability Theory, p22): #1 -> #6  (dropped OUT of the top 3)
  chunk 874 (Readings from Hindi Literature, p228): #9 -> #2  (promoted INTO the top 3)
  ★ gold chunk is NOT in the bi-encoder top 10 (full rank: #325). Re-ranking cannot fix a recall failure.


**Answer 1 (baseline top 3):** The provided passages do not contain the answer.

**Answer 2 (re-ranked top 3):** The provided passages do not contain the answer.

## 11. Latency profiling

We time **retrieval only**, as the assignment asks (baseline retrieval vs the re-ranking pipeline). The LLM is timed separately, because it is the same cost for both pipelines and would hide the difference.

Timing rules for a fair measurement:
- **warm-up runs first**, because the first call on a GPU is slow (loading kernels);
- `torch.cuda.synchronize()` before reading the clock, because GPU work runs in the background and the clock would otherwise stop too early;
- **average over many runs and all bank queries**.

In [19]:
def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def time_ms(fn, repeats=10, warmup=2):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        sync(); t = time.perf_counter()
        fn()
        sync(); times.append((time.perf_counter() - t) * 1000)
    return np.mean(times)

queries = [q for _, q, _, _ in QUERY_BANK]
t_base, t_rr_only, t_p2 = [], [], []
for q in queries:
    cands = baseline_retrieve(q)
    t_base.append(time_ms(lambda: baseline_retrieve(q)))                 # Pipeline 1 retrieval
    t_rr_only.append(time_ms(lambda: rerank(q, cands)))                  # the extra re-ranking step
    t_p2.append(time_ms(lambda: rerank(q, baseline_retrieve(q))))        # Pipeline 2 retrieval

# LLM generation time, measured a few times (it is the same for both pipelines)
t_llm = []
for q in queries[:3]:
    hits = baseline_retrieve(q)[:TOP_K]
    sync(); t = time.perf_counter(); answer(q, hits); sync()
    t_llm.append((time.perf_counter() - t) * 1000)

latency = pd.DataFrame([
    {"stage": "Pipeline 1: embed query + score all chunks + top 10", "mean ms": np.mean(t_base)},
    {"stage": "Re-ranking step only: cross-encoder on 10 pairs",      "mean ms": np.mean(t_rr_only)},
    {"stage": "Pipeline 2: baseline retrieval + re-ranking",          "mean ms": np.mean(t_p2)},
    {"stage": "LLM answer generation (same for both)",                "mean ms": np.mean(t_llm)},
]).round(1)
display(latency)
print(f"Re-ranking makes retrieval {np.mean(t_p2) / np.mean(t_base):.1f}x slower, "
      f"but adds only {np.mean(t_p2) - np.mean(t_base):.0f} ms next to ~{np.mean(t_llm) / 1000:.1f} s of LLM time.")
print(f"(device: {DEVICE}, {len(chunks)} chunks, {len(queries)} queries x 10 repeats)")

,stage,mean ms
0,Pipeline 1: embed query + score all chunks + top 10,9.6
1,Re-ranking step only: cross-encoder on 10 pairs,31.9
2,Pipeline 2: baseline retrieval + re-ranking,43.2
3,LLM answer generation (same for both),2714.4


Re-ranking makes retrieval 4.5x slower, but adds only 34 ms next to ~2.7 s of LLM time.
(device: cuda, 1128 chunks, 31 queries x 10 repeats)


### 11.1 Evidence for the scaling question: what if we cross-encoded *everything*?

Here we actually run the cross-encoder on **all** chunks for one query and compare it with the bi-encoder cost. Then we extrapolate to a large collection. This turns slide 28's "one forward pass **per candidate**" into real numbers from your own run.

In [ ]:
q = queries[0]
sync(); t = time.perf_counter()
_ = reranker.predict([(q, c["text"]) for c in chunks], batch_size=64, show_progress_bar=False)
sync(); t_ce_all = (time.perf_counter() - t) * 1000

per_pair = t_ce_all / len(chunks)
scale = pd.DataFrame([
    {"approach": f"bi-encoder retrieval over {len(chunks)} chunks", "ms per query": np.mean(t_base)},
    {"approach": f"cross-encoder on top {N_CANDIDATES} only (Pipeline 2 extra cost)", "ms per query": np.mean(t_rr_only)},
    {"approach": f"cross-encoder on ALL {len(chunks)} chunks", "ms per query": t_ce_all},
    {"approach": "cross-encoder on 1,000,000 chunks (extrapolated)", "ms per query": per_pair * 1_000_000},
]).round(1)
display(scale)
print(f"cross-encoder cost: about {per_pair:.2f} ms per (query, chunk) pair; "
      f"1M chunks would take about {per_pair * 1_000_000 / 60000:.1f} minutes for ONE query.")

,approach,ms per query
0,bi-encoder retrieval over 1128 chunks,7.3
1,cross-encoder on top 10 only (Pipeline 2 extra cost),28.8
2,cross-encoder on ALL 1128 chunks,3432.4
3,"cross-encoder on 1,000,000 chunks (extrapolated)",3042892.2


cross-encoder cost: about 3.04 ms per (query, chunk) pair; 1M chunks would take about 50.7 minutes for ONE query.
